# Overview

In this tutorial, we will go over how to carry out some basic molecular dynamics (MD) simulations using MLIPs. These MLIPs, in principle, can act as fast surrogates for density functional theory (DFT) and therefore can mimic an ab initio MD (AIMD) simulation at a very small fraction of the cost. Of course, in practice, they are ML models and therefore the agreement with AIMD is something that is not guaranteed.

Disclaimer: This is not an MD course, and there are _many_ important aspects one needs to know to run an MD simulation well enough to produce high-quality results. That said, hopefully this gives you a high level understanding of what's going on.

We will perform an MD simulation on the perovskite known as [methyl ammonium lead iodide](https://en.wikipedia.org/wiki/Methylammonium_lead_halide) (MAPbI3 for short), one of the first materials used in perovskite-based solar cells. The perovskite structure has [a cation in its void space](https://upload.wikimedia.org/wikipedia/commons/d/df/CH3NH3PbI3_structure.png), which we will watch jiggle around during the MD simulation.


# Documentation

The ASE documentation has a specific section for the [molecular dynamics methods](https://ase-lib.org/ase/md.html). There is also a brief tutorial in the [ASE 2023 Workshop](https://ase-workshop-2023.github.io/tutorial/10-dynamics/index.html).


# Setup


In [ ]:
!uv pip install "ase>=3.28.0" "matcalc @ git+https://github.com/materialyzeai/matcalc.git" "upet>=0.2.2"

In [ ]:
!curl -o "MAPbI3.cif" "https://www.crystallography.net/cod/4124388.cif" 

# Creating the Atoms Object


We will start by reading in the CIF we downloaded in the prior step.


In [ ]:
from ase.io import read
from ase.visualize import view

atoms = read("MAPbI3.cif")
view(atoms, viewer="x3d")

The first thing we should do before setting up an MD simulation is to see how large our unit cell is. We do this because we want to make sure the simulation cell does not have ficticious self interactions across the periodic boundary conditions. Many MLIPs have a cutoff radius of 5 or 6 Å, but this depends on the model.


In [ ]:
atoms.cell.lengths()

For good measure and for demonstration purposes, let's just double the a and b lattice dimensions.


In [ ]:
atoms *= (2, 2, 1)

# Defining the Calculator


Alright, now we must pick an ASE `Calculator` that defines the energies, forces, and stresses. For this demonstration, we will use one of the [UPET foundation models](https://github.com/lab-cosmo/upet), specifically the `upet-mad-xs` model because it is (as the name suggests) extra small and therefore extra fast (at the expense of lower accuracy).


In [ ]:
from upet.calculator import UPETCalculator
calc = UPETCalculator(
    model="pet-mad-xs",# choice of MLIP
                       device="cpu", # switch to "cuda" for GPU
                       )

# Matcalc

We will start this demonstration by using `matcalc` to carry out the MD simulations. As a reminder, `matcalc` is just a wrapper around ASE and provides some convenient utility functions for various simulation tasks. It is not very flexible, so for advanced usage, it is better to use ASE directly, which we will do afterwards.


For this simulation, we are going to run an `NVT` simulation, meaning that the number of atoms (N), volume (V), and temperature (T) are held constant. In practice, the temperature is not held exactly constant because we use a "thermostat" that tries to maintain an average temperature, typically by rescaling particle velocities. That said, it's close enough. If you wanted to allow the unit cell volume to change, that would be an `NPT` simulation, which would require you to specify a constant pressure as well (as controlled by a "barostat"). For now, we will just carry out an NVT simulation for simplicity.


In [ ]:
from matcalc import MDCalc
from ase.units import bar

MDCalc(
    calc, # your ASE calculator
    ensemble="nvt", # one of many possible MD ensembles and methods
    steps=1000, # number of steps to run the MD simulation
    timestep=1, # timestep (in fs); 1 fs is typically a reasonable default  
    temperature=300.0, # user-defined temperature in K
    pressure=1 * bar, # user-defined pressure (ignored in NVT!)
    trajfile="md.traj", # write out ASE GUI-compatible trajectory file
    logfile="md.log", # write out some metadata to a log file
    relax_structure=True, # relax the structure before the MD simulation
    fmax=0.05, # eV/Å tolerance for the relaxation
    set_com_stationary=True, # ensure the center of mass is not moving to start
).calc(atoms)

Our simulation above only ran for 1000 fs (i.e. 1 ps). That is pretty short. In practice, you will typically want to simulate several nanoseconds at minimum, but you get the idea.


Now let's view the MD trajectory! You should be able to see the central ammonium cation jiggling around during the MD trajectory. This is something that would be difficult to observe from a standard structure relaxation at 0 K, as finite temperature imparts some thermal motion on the atoms and causes the atoms to move about their minimum energy configuration.

Note: In the ASE GUI, going to `View > Show Bonds` will make the simulation a bit easier to understand. This will draw bonds between atoms based on a geometry-based heuristic. There is no specific bonding information in the MLIP or in DFT itself.


In [ ]:
view("md.traj", index=":")

If you open `md.log`, you'll also see some information about the energy, temperature, and other properties as a function of the MD simulation length. Note that the temeprature will not be perfectly constant, as described above. There are optional keyword arguments in `MDCalc` that can be specified, which will control the time with which the thermostat rescales the velocities, but that is not important here.


# ASE

If you don't think you'll be doing MD simulations much, then you can stop here (feel free to change some of the parameters in the above `MDCalc` run though!).

In this part of the demo, we'll do the same analysis but using ASE directly, which gives you a bit more freedom.


First, let's redefine our `Atoms` object


In [ ]:
atoms = read("MAPbI3.cif") * (2,2,1)
atoms.calc = calc

Now we will relax the structure before running the MD simulation.


In [ ]:
from ase.optimize import BFGS

BFGS(atoms).run(fmax=0.05)

With our updated `Atoms` object, it's time to run the MD simulation. The process is not terribly different from before, but we must do some things manually.


We start by giving the atoms a distribution of velocities to match the desired temperature. We then make sure the center of mass of the structure isn't mobile, otherwise it will just float across the simulation cell, which is annoying.


In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.velocitydistribution import Stationary

MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)
Stationary(atoms)

Now we need to pick an ensemble and method as outlined in the [ASE documentation](https://ase-lib.org/ase/md.html). Here, we will pack one of the [Constant NVT simulations](https://ase-lib.org/ase/md.html#constant-nvt-simulations-the-canonical-ensemble), of which ASE recommends several. One of the recommended ones is the Nosé-Hoover chain, which again is outside the scope of this course, but was the default when specifying `"nvt"` in `matcalc`. We will do that here. Note that the units are a bit funky.

For full details on the different keyword arguments, refer to the ASE documentation.


In [ ]:
from ase.units import fs
from ase.md.nose_hoover_chain import NoseHooverChainNVT

dyn = NoseHooverChainNVT(
    atoms,
    timestep=1.0 * fs,
    temperature_K=1000.0,
    tdamp=100 * fs,
    trajectory="md2.traj",
    logfile="md2.log",
)
dyn.run(steps=1000)

And we can once again view the trajectory.


In [ ]:
view("md2.traj", index=":")

This process is effectively the same as what was done when using `matcalc`, but it gives you more flexibility if needed.
